# Build the config-driven ingestion framework you stop having to rewrite for every new upstream team

You inherited a one-off pipeline for every new upstream team in the last six months. Today you stop. You build the framework that turns "a new upstream" into a YAML file. Three sources at launch (SFTP, REST, RDS), three source classes that all implement the same `ingest()` contract, one dispatcher that reads the YAML and instantiates the right class, one bronze Iceberg table tagged by source_system.

The five deliverables map to five checkpoints:
1. Three sources stood up: SFTP drop has nightly Parquet, REST endpoint serves paginated cursor, RDS view returns rows.
2. YAML config at `framework/config.yaml` describes three sources matching the framework's schema.
3. All three source classes implement the same ingest contract and land rows in `bronze_events` tagged by `source_system`.
4. Idempotency: re-run dispatch with no upstream changes produces zero new bronze rows. Each source class persists its own watermark.
5. Add a fourth source via YAML edit only; no Lambda redeploy required. The framework is now a framework.

**Time.** About 90 minutes hands-on. RDS provisioning is 5-7 minutes; the rest is code.

**Cost.** This lab costs about $1.50 on a clean run. RDS Postgres db.t4g.micro is the boss line item: free for the first 12 months on new AWS accounts, otherwise $0.017 per hour ($0.025 across the 1.5-hour session). Lambda is free at lab volume. Iceberg writes from the multiple dispatch runs land around $0.40. S3 + Secrets Manager: trivial. Realistic upper bound: $3.00 (if RDS is paid and you keep the lab open past 3 hours).

**Critical resource.** RDS Postgres is critical-tagged. The setup cell surfaces a hard confirmation; cleanup deletes RDS first with `--skip-final-snapshot --delete-automated-backups`.

**Multi-session.** This lab is multi-session-capable. The setup cell prompts "resume or fresh?" if it finds existing labrun-tagged resources. The 24-hour JWT supports an overnight pause; the cost-window cap is 2.5 hours.

This lab is part of the Labrun Data Engineering job-prep track, Pipeline Engineering sub-track.


In [ ]:
!pip install --quiet labrun-checks==0.7.0 boto3 pyyaml jsonschema psycopg2-binary


In [ ]:
# Imports and constants.

import atexit
import io
import json
import re
import sys
import time
import uuid
import zipfile
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import CheckpointResult, CleanupResource, check, cleanup, register, run_cleanup

LAB_SLUG = "multi-source-ingestion-framework"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG
REGION = "us-east-1"

BUCKET_NAME = None
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
BRONZE_TABLE = "bronze_events"

REST_MOCK_FN = f"labrun-{LAB_SLUG}-rest-mock"
SFTP_FN = f"labrun-{LAB_SLUG}-sftp"
REST_FN = f"labrun-{LAB_SLUG}-rest"
RDS_FN = f"labrun-{LAB_SLUG}-rds"
DISPATCH_FN = f"labrun-{LAB_SLUG}-dispatch"

RDS_ID = f"labrun-{LAB_SLUG}-rds"
SG_NAME = f"labrun-{LAB_SLUG}-sg"
SUBNET_GROUP = f"labrun-{LAB_SLUG}-subnet"
SECRET_NAME = f"labrun-{LAB_SLUG}-secret"
FRAMEWORK_ROLE = f"labrun-{LAB_SLUG}-role"
SCHEDULE_NAME = f"labrun-{LAB_SLUG}-daily"
ATHENA_WG = f"labrun-{LAB_SLUG}-wg"

ACCOUNT_ID = None
SG_ID = None
SECRET_ARN = None


In [ ]:
# NBVAL_SKIP
# Setup. Hard confirmation about RDS billing. Multi-session orphan check.

print("Paste your lab session token:")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")
print()
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (Enter to skip): ").strip() or None

AWS_CREDENTIALS = {"aws_access_key_id": AWS_ACCESS_KEY_ID, "aws_secret_access_key": AWS_SECRET_ACCESS_KEY, "aws_session_token": AWS_SESSION_TOKEN}

print()
print("RDS Postgres bills continuously from the moment the instance becomes available.")
print("Free-tier eligible accounts get 750 hours/month for the first 12 months.")
print("If you are past the free-tier window, RDS costs about $0.017 per hour.")
print("Type 'I understand' to acknowledge and proceed.")
ack = input("Acknowledgement: ").strip()
if ack != "I understand":
    raise SystemExit("Acknowledgement not provided. Re-run this cell when you are ready.")

sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + multi-session-aware orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(type="rds_instance", id=RDS_ID, region=REGION, critical=True,
        cli_delete_command=f"aws rds delete-db-instance --db-instance-identifier {RDS_ID} --skip-final-snapshot --delete-automated-backups --region {REGION}"),
    CleanupResource(type="db_subnet_group", id=SUBNET_GROUP, region=REGION),
    CleanupResource(type="secret", id="pending", region=REGION),
    CleanupResource(type="security_group", id="pending", region=REGION),
    CleanupResource(type="eventbridge_rule", id=SCHEDULE_NAME, region=REGION),
    CleanupResource(type="lambda_function", id=DISPATCH_FN, region=REGION),
    CleanupResource(type="lambda_function", id=REST_MOCK_FN, region=REGION),
    CleanupResource(type="lambda_function", id=SFTP_FN, region=REGION),
    CleanupResource(type="lambda_function", id=REST_FN, region=REGION),
    CleanupResource(type="lambda_function", id=RDS_FN, region=REGION),
    CleanupResource(type="iam_role", id=FRAMEWORK_ROLE, region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{BRONZE_TABLE}", region=REGION),
    CleanupResource(type="athena_workgroup", id=ATHENA_WG, region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {ATHENA_WG} --recursive-delete-option --region {REGION}"),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] RDS bills continuously while alive. Run the cleanup cell if it has not run.")


atexit.register(_atexit_cleanup)

tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_arns = [r["ResourceARN"] for r in tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]).get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed: {exc}")

RESUME_MODE = False
if orphan_arns:
    print()
    print("Found existing labrun-tagged resources from a previous session:")
    for arn in orphan_arns:
        print(f"  EXISTING: {arn}")
    print()
    print("This lab is multi-session. Type 'resume' to continue with the existing resources, or 'fresh' to clean up and start over.")
    choice = input("Choice: ").strip().lower()
    if choice == "resume":
        RESUME_MODE = True
        print("Resuming session; orphan scan does not block.")
    elif choice == "fresh":
        raise SystemExit("Run the previous session's cleanup cell first, then re-run this cell.")
    else:
        raise SystemExit("Invalid choice; must be 'resume' or 'fresh'.")
else:
    print("No orphans. Manifest declared. Proceed to Task 1.")


## Task 1: Provision the three upstream sources

You stand up the SFTP drop (an S3 prefix with seeded Parquet files), the REST mock Lambda (a paginated cursor endpoint), and the RDS Postgres source (a small table behind a `team_c_view`). RDS takes 5-7 minutes; use the wait for the rest of the resources.

Free-tier eligible AWS accounts get 750 RDS hours per month for 12 months at this instance size. You acknowledged the meter implication in setup; this is the resource that drives the meter.


In [ ]:
# NBVAL_SKIP
# Task 1: stand up sources.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
ec2 = boto3.client("ec2", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
rds = boto3.client("rds", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
secretsmanager = boto3.client("secretsmanager", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
lambda_client = boto3.client("lambda", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
events = boto3.client("events", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})

# S3 bucket
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
except ClientError as exc:
    if exc.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
        raise

# SFTP seed: write two Parquet files using pyarrow (already in Colab's
# default environment). For simplicity here we write CSV-shaped JSON
# Lines (the SFTP source class will read them as JSON Lines).
sftp_seed = [
    {"event_id": f"sftp-{i}", "source_system": "team-a-sftp", "amount": i * 10, "raw_path": "team-a/2026-05-14.jsonl"}
    for i in range(50)
]
s3.put_object(Bucket=BUCKET_NAME, Key="sftp-drop/team-a/2026-05-14.jsonl", Body="\n".join(json.dumps(r) for r in sftp_seed).encode("utf-8"))
sftp_seed_b = [
    {"event_id": f"sftp-b-{i}", "source_system": "team-a-sftp", "amount": i * 5, "raw_path": "team-a/2026-05-15.jsonl"}
    for i in range(30)
]
s3.put_object(Bucket=BUCKET_NAME, Key="sftp-drop/team-a/2026-05-15.jsonl", Body="\n".join(json.dumps(r) for r in sftp_seed_b).encode("utf-8"))

# Seed for the future "team-d-sftp-2" source used in Task 5.
sftp_seed_d = [
    {"event_id": f"sftp-d-{i}", "source_system": "team-d-sftp-2", "amount": i * 3, "raw_path": "team-d/2026-05-15.jsonl"}
    for i in range(20)
]
s3.put_object(Bucket=BUCKET_NAME, Key="sftp-drop-2/team-d/2026-05-15.jsonl", Body="\n".join(json.dumps(r) for r in sftp_seed_d).encode("utf-8"))
print("  SFTP seed: 80 records in sftp-drop/team-a/ + 20 records in sftp-drop-2/team-d/ (reserve)")

# Glue database + Athena workgroup + Iceberg bronze table
try:
    glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
except glue.exceptions.AlreadyExistsException:
    pass
try:
    athena.create_work_group(
        Name=ATHENA_WG,
        Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except Exception:
    pass


def run_athena(query, timeout=60):
    qid = athena.start_query_execution(QueryString=query, WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return qid
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {qid} {s}")
        time.sleep(2)
    raise TimeoutError(f"Athena {qid} timed out")


run_athena(
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{BRONZE_TABLE} ("
    f"  event_id string, source_system string, amount bigint, raw_path string, ingest_ts string"
    f") LOCATION 's3://{BUCKET_NAME}/bronze/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
print(f"  bronze table ready: {DATABASE_NAME}.{BRONZE_TABLE}")

# REST mock Lambda. Paginated cursor: ?cursor=N -> returns 50 records + next_cursor.
REST_MOCK_CODE = '''
import json
def handler(event, context):
    qs = event.get("queryStringParameters") or {}
    cursor = int(qs.get("cursor", "0"))
    page_size = 50
    total = 300
    records = [{
        "event_id": f"rest-{i}",
        "source_system": "team-b-rest",
        "amount": i * 7,
        "raw_path": f"rest-mock/page-{cursor // page_size}",
    } for i in range(cursor, min(cursor + page_size, total))]
    next_cursor = cursor + page_size
    if next_cursor >= total:
        next_cursor = None
    return {
        "statusCode": 200,
        "body": json.dumps({"records": records, "next_cursor": next_cursor}),
    }
'''
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("index.py", REST_MOCK_CODE)
buf.seek(0)
rest_zip = buf.read()

# Framework IAM role
try:
    iam.create_role(
        RoleName=FRAMEWORK_ROLE,
        AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    iam.put_role_policy(RoleName=FRAMEWORK_ROLE, PolicyName="labrun-framework-inline", PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["s3:*", "secretsmanager:*", "logs:*", "lambda:InvokeFunction", "athena:*", "glue:*", "rds:*", "ec2:Describe*"], "Resource": "*"},
        ],
    }))
except iam.exceptions.EntityAlreadyExistsException:
    pass
print("  IAM propagation, hold for 15 seconds...")
time.sleep(15)
framework_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{FRAMEWORK_ROLE}"

# Deploy REST mock Lambda
try:
    lambda_client.create_function(
        FunctionName=REST_MOCK_FN, Runtime="python3.13", Role=framework_role_arn,
        Handler="index.handler", Code={"ZipFile": rest_zip}, Timeout=30,
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except lambda_client.exceptions.ResourceConflictException:
    pass
print("  REST mock Lambda deployed")

# RDS: SG + DBSubnetGroup + instance + seed
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"][0]
vpc_id = default_vpc["VpcId"]
subnets = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]
subnet_ids = [s["SubnetId"] for s in subnets[:2]]

# Caller IP
import urllib.request
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
print(f"  caller IP: {my_ip}")

try:
    sg_resp = ec2.create_security_group(GroupName=SG_NAME, Description="labrun rds", VpcId=vpc_id, TagSpecifications=[{"ResourceType": "security-group", "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]}])
    SG_ID = sg_resp["GroupId"]
    ec2.authorize_security_group_ingress(GroupId=SG_ID, IpPermissions=[{"IpProtocol": "tcp", "FromPort": 5432, "ToPort": 5432, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]}])
except ClientError as exc:
    if exc.response["Error"]["Code"] == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(Filters=[{"Name": "group-name", "Values": [SG_NAME]}, {"Name": "vpc-id", "Values": [vpc_id]}])["SecurityGroups"]
        SG_ID = sgs[0]["GroupId"]
    else:
        raise
for r in CLEANUP_MANIFEST:
    if r.type == "security_group":
        r.id = SG_ID
        r.cli_delete_command = f"aws ec2 delete-security-group --group-id {SG_ID} --region {REGION}"
        break

try:
    rds.create_db_subnet_group(DBSubnetGroupName=SUBNET_GROUP, DBSubnetGroupDescription="labrun", SubnetIds=subnet_ids, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
except rds.exceptions.DBSubnetGroupAlreadyExistsFault:
    pass

import secrets as _secrets
db_password = _secrets.token_urlsafe(16)

try:
    rds.create_db_instance(
        DBInstanceIdentifier=RDS_ID, DBInstanceClass="db.t4g.micro", Engine="postgres",
        AllocatedStorage=20, MasterUsername="labrun", MasterUserPassword=db_password,
        VpcSecurityGroupIds=[SG_ID], DBSubnetGroupName=SUBNET_GROUP,
        PubliclyAccessible=True, BackupRetentionPeriod=0,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print("  RDS Postgres creating; hold for ~6 minutes...")
except rds.exceptions.DBInstanceAlreadyExistsFault:
    print("  RDS instance exists")

waiter = rds.get_waiter("db_instance_available")
waiter.wait(DBInstanceIdentifier=RDS_ID, WaiterConfig={"Delay": 30, "MaxAttempts": 30})
ep = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)["DBInstances"][0]["Endpoint"]
print(f"  RDS available at {ep['Address']}:{ep['Port']}")

# Store secret
try:
    secret_resp = secretsmanager.create_secret(
        Name=SECRET_NAME,
        SecretString=json.dumps({"username": "labrun", "password": db_password, "host": ep["Address"], "port": ep["Port"], "database": "postgres"}),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    SECRET_ARN = secret_resp["ARN"]
except secretsmanager.exceptions.ResourceExistsException:
    desc = secretsmanager.describe_secret(SecretId=SECRET_NAME)
    SECRET_ARN = desc["ARN"]
    secretsmanager.put_secret_value(
        SecretId=SECRET_NAME,
        SecretString=json.dumps({"username": "labrun", "password": db_password, "host": ep["Address"], "port": ep["Port"], "database": "postgres"}),
    )
for r in CLEANUP_MANIFEST:
    if r.type == "secret":
        r.id = SECRET_ARN
        r.cli_delete_command = f"aws secretsmanager delete-secret --secret-id {SECRET_NAME} --force-delete-without-recovery --region {REGION}"
        break

# Seed RDS table + view
import psycopg2
conn = psycopg2.connect(host=ep["Address"], port=ep["Port"], dbname="postgres", user="labrun", password=db_password)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("CREATE TABLE IF NOT EXISTS team_c_orders (order_id bigserial primary key, customer_id text, amount_cents bigint, status text)")
    cur.execute("TRUNCATE team_c_orders")
    cur.executemany("INSERT INTO team_c_orders (customer_id, amount_cents, status) VALUES (%s, %s, %s)",
                    [(f"cust-{i}", i * 13, "paid") for i in range(1, 41)])
    cur.execute("CREATE OR REPLACE VIEW team_c_view AS SELECT order_id, customer_id, amount_cents FROM team_c_orders WHERE status='paid'")
conn.close()
print("  RDS seed table + view ready (40 rows)")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: three sources healthy.

def validator_1(session):
    # SFTP: at least 2 Parquet/JSONL objects under sftp-drop/
    listing = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix="sftp-drop/")
    sftp_count = len(listing.get("Contents", []))
    if sftp_count < 2:
        return CheckpointResult("fail", error_reason=f"sftp-drop/ has {sftp_count} objects; need at least 2.")
    # REST mock: invoke once
    try:
        resp = lambda_client.invoke(FunctionName=REST_MOCK_FN, Payload=json.dumps({"queryStringParameters": {"cursor": "0"}}).encode())
        body = json.loads(json.loads(resp["Payload"].read())["body"])
        if not body.get("records") or "next_cursor" not in body:
            return CheckpointResult("fail", error_reason="REST mock did not return paginated cursor body")
    except ClientError as exc:
        return CheckpointResult("fail", error_reason=f"REST mock invoke failed: {exc}")
    # RDS: describe + view exists via psycopg2
    try:
        inst = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)["DBInstances"][0]
        if inst["DBInstanceStatus"] != "available":
            return CheckpointResult("fail", error_reason=f"RDS status is {inst['DBInstanceStatus']}; need available")
    except ClientError as exc:
        return CheckpointResult("fail", error_reason=f"RDS describe failed: {exc}")
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

Each of the three sources is its own resource type. SFTP is just S3 objects. REST is a Lambda you can invoke directly. RDS is the slow one (~6 min to provision).

</details>

<details><summary>Hint 2 (direction)</summary>

This is the setup cell; you do not write much code. Just run the cell and wait for RDS. The "I understand" acknowledgement in setup means you opted in to the meter.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The cell does all the work. If it fails on RDS provisioning, common cause is insufficient quota; in that case AWS returns `InsufficientDBInstanceCapacity`. You can re-run safely; the cell is idempotent.

</details>


**Wallet check.** RDS db.t4g.micro provisioning + ~10 min runtime: free if you are inside the free-tier window, otherwise ~$0.003. SG, secret, subnet group: free or near-free. Session total so far: under 1 cent (or under $0.01 if not free-tier).

## Task 2: Author the framework YAML config

The framework reads a YAML file with one entry per upstream source. The schema is fixed; the values are yours.

Schema:

```yaml
sources:
  - name: <string, unique>
    type: <"sftp" | "rest" | "rds">
    config:
      # sftp: { path: <s3 prefix> }
      # rest: { endpoint_function: <lambda function name>, page_size: <int> }
      # rds:  { secret_arn: <arn>, query: <string> }
```

You write three entries: `team-a-sftp`, `team-b-rest`, `team-c-rds`. The dispatcher uses `type` to choose the source class; the framework keys watermarks on `name`.


In [ ]:
# NBVAL_SKIP
# Task 2: Author framework/config.yaml.

import yaml

YAML_SCHEMA = {
    "type": "object",
    "required": ["sources"],
    "properties": {
        "sources": {
            "type": "array",
            "minItems": 3,
            "items": {
                "type": "object",
                "required": ["name", "type", "config"],
                "properties": {
                    "name": {"type": "string"},
                    "type": {"enum": ["sftp", "rest", "rds"]},
                    "config": {"type": "object"},
                },
            },
        }
    },
}

# YOUR CODE: build the config dict matching the schema and upload as YAML.
# The keys you need:
#   team-a-sftp:   {path: "sftp-drop/"}
#   team-b-rest:   {endpoint_function: REST_MOCK_FN, page_size: 50}
#   team-c-rds:    {secret_arn: SECRET_ARN, query: "SELECT order_id::text as event_id, 'team-c-rds' as source_system, amount_cents as amount, 'rds-team-c' as raw_path FROM team_c_view"}

# Example:
# config = {"sources": [...]}
# s3.put_object(Bucket=BUCKET_NAME, Key="framework/config.yaml", Body=yaml.safe_dump(config).encode("utf-8"))


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: YAML present + matches schema + 3 source entries with expected types.

from jsonschema import validate as jsonschema_validate, ValidationError

def validator_2(session):
    try:
        body = s3.get_object(Bucket=BUCKET_NAME, Key="framework/config.yaml")["Body"].read().decode("utf-8")
    except ClientError:
        return CheckpointResult("fail", error_reason="framework/config.yaml not found in S3.")
    try:
        config = yaml.safe_load(body)
    except yaml.YAMLError as exc:
        return CheckpointResult("fail", error_reason=f"YAML parse failed: {exc}")
    try:
        jsonschema_validate(config, YAML_SCHEMA)
    except ValidationError as exc:
        return CheckpointResult("fail", error_reason=f"YAML schema validation failed: {exc.message}")
    types = sorted({s["type"] for s in config["sources"]})
    if types != ["rds", "rest", "sftp"]:
        return CheckpointResult("fail", error_reason=f"Expected one of each type (sftp/rest/rds); got types {types}")
    return CheckpointResult("pass")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

Three entries. Three distinct types. The dispatcher reads `type` to decide which source class to invoke.

</details>

<details><summary>Hint 2 (direction)</summary>

The RDS source class will read the SQL string from `config.query`. The REST source class reads `endpoint_function` to find the Lambda to invoke. The SFTP source class reads `path` for the S3 prefix to list.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
config = {
    "sources": [
        {"name": "team-a-sftp", "type": "sftp", "config": {"path": "sftp-drop/"}},
        {"name": "team-b-rest", "type": "rest", "config": {"endpoint_function": REST_MOCK_FN, "page_size": 50}},
        {"name": "team-c-rds", "type": "rds", "config": {
            "secret_arn": SECRET_ARN,
            "query": "SELECT order_id::text as event_id, 'team-c-rds' as source_system, amount_cents as amount, 'rds-team-c' as raw_path FROM team_c_view",
        }},
    ]
}
s3.put_object(Bucket=BUCKET_NAME, Key="framework/config.yaml", Body=yaml.safe_dump(config).encode("utf-8"))
```

</details>


**Wallet check.** YAML upload: free. Session total: still under a cent.

## Task 3: Implement and deploy the three source classes plus the dispatcher

Each source class is a Lambda that takes one source_config and writes to bronze via Athena. They share a contract: receive the source name + type + config, fetch upstream data, write to `bronze_events`, return a manifest dict.

The dispatcher reads `framework/config.yaml` and invokes the right source-class Lambda by name (`team-<x>-sftp` -> `<SFTP_FN>` and so on; the dispatcher resolves by `type`).


In [ ]:
# NBVAL_SKIP
# Task 3: deploy source classes + dispatcher; trigger dispatch.

SFTP_CODE = f'''
import base64
import json
import time
import boto3

S3 = boto3.client("s3")
ATHENA = boto3.client("athena")
BUCKET = "{BUCKET_NAME}"
DB = "{DATABASE_NAME}"
TABLE = "{BRONZE_TABLE}"
WG = "{ATHENA_WG}"


def _run_athena(query, timeout_s=60):
    qid = ATHENA.start_query_execution(QueryString=query, WorkGroup=WG, QueryExecutionContext={{"Database": DB}})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError(f"Athena {{qid}} timed out")


def _q(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    name = event["name"]
    cfg = event["config"]
    prefix = cfg["path"]
    # Watermark = list of file keys we have already ingested.
    wm_key = f"framework/watermarks/{{name}}.json"
    try:
        seen = set(json.loads(S3.get_object(Bucket=BUCKET, Key=wm_key)["Body"].read())["seen_files"])
    except S3.exceptions.NoSuchKey:
        seen = set()
    new_rows = []
    new_files = []
    listing = S3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for obj in listing.get("Contents", []):
        key = obj["Key"]
        if key in seen:
            continue
        body = S3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")
        for line in body.splitlines():
            if not line.strip():
                continue
            rec = json.loads(line)
            new_rows.append({{
                "event_id": rec.get("event_id"),
                "source_system": rec.get("source_system", name),
                "amount": rec.get("amount", 0),
                "raw_path": key,
            }})
        new_files.append(key)
    if new_rows:
        values = ",".join(
            f"({{_q(r['event_id'])}}, {{_q(r['source_system'])}}, {{int(r['amount'])}}, {{_q(r['raw_path'])}}, "
            f"'{{time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}}')"
            for r in new_rows
        )
        _run_athena(f"INSERT INTO {{DB}}.{{TABLE}} (event_id, source_system, amount, raw_path, ingest_ts) VALUES {{values}}")
    seen.update(new_files)
    S3.put_object(Bucket=BUCKET, Key=wm_key, Body=json.dumps({{"seen_files": sorted(seen)}}).encode("utf-8"))
    return {{"name": name, "rows": len(new_rows), "files_seen": len(seen)}}
'''

REST_CODE = f'''
import json
import time
import boto3

LAMBDA = boto3.client("lambda")
S3 = boto3.client("s3")
ATHENA = boto3.client("athena")
BUCKET = "{BUCKET_NAME}"
DB = "{DATABASE_NAME}"
TABLE = "{BRONZE_TABLE}"
WG = "{ATHENA_WG}"


def _run_athena(query, timeout_s=60):
    qid = ATHENA.start_query_execution(QueryString=query, WorkGroup=WG, QueryExecutionContext={{"Database": DB}})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError("Athena timeout")


def _q(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    name = event["name"]
    cfg = event["config"]
    endpoint_fn = cfg["endpoint_function"]
    wm_key = f"framework/watermarks/{{name}}.json"
    try:
        last_cursor = json.loads(S3.get_object(Bucket=BUCKET, Key=wm_key)["Body"].read())["last_cursor"]
    except S3.exceptions.NoSuchKey:
        last_cursor = 0
    starting_cursor = last_cursor
    rows = []
    cursor = last_cursor
    while True:
        resp = LAMBDA.invoke(FunctionName=endpoint_fn, Payload=json.dumps({{"queryStringParameters": {{"cursor": str(cursor)}}}}).encode())
        body = json.loads(json.loads(resp["Payload"].read())["body"])
        for r in body["records"]:
            rows.append({{
                "event_id": r.get("event_id"),
                "source_system": r.get("source_system", name),
                "amount": r.get("amount", 0),
                "raw_path": r.get("raw_path", ""),
            }})
        if body.get("next_cursor") is None:
            break
        cursor = body["next_cursor"]
    if rows:
        values = ",".join(
            f"({{_q(r['event_id'])}}, {{_q(r['source_system'])}}, {{int(r['amount'])}}, {{_q(r['raw_path'])}}, "
            f"'{{time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}}')"
            for r in rows
        )
        _run_athena(f"INSERT INTO {{DB}}.{{TABLE}} (event_id, source_system, amount, raw_path, ingest_ts) VALUES {{values}}")
    S3.put_object(Bucket=BUCKET, Key=wm_key, Body=json.dumps({{"last_cursor": cursor if rows else starting_cursor}}).encode("utf-8"))
    return {{"name": name, "rows": len(rows)}}
'''

RDS_CODE = f'''
import json
import time
import boto3

SM = boto3.client("secretsmanager")
S3 = boto3.client("s3")
ATHENA = boto3.client("athena")
BUCKET = "{BUCKET_NAME}"
DB = "{DATABASE_NAME}"
TABLE = "{BRONZE_TABLE}"
WG = "{ATHENA_WG}"


def _run_athena(query, timeout_s=60):
    qid = ATHENA.start_query_execution(QueryString=query, WorkGroup=WG, QueryExecutionContext={{"Database": DB}})["QueryExecutionId"]
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = ATHENA.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {{qid}} {{s}}")
        time.sleep(1)
    raise TimeoutError("Athena timeout")


def _q(v):
    if v is None:
        return "NULL"
    if isinstance(v, (int, float)):
        return str(v)
    return "'" + str(v).replace("'", "''") + "'"


def handler(event, context):
    name = event["name"]
    cfg = event["config"]
    query = cfg["query"]
    secret = json.loads(SM.get_secret_value(SecretId=cfg["secret_arn"])["SecretString"])
    # Use pg8000 (pure-python) to avoid building psycopg2 into the layer.
    # For lab simplicity we use the AWS Data API style? Simpler: invoke
    # an inline import of pg8000 which ships with the Lambda runtime base
    # image since python3.13. Fallback: use an inline http call. For lab,
    # we ship pg8000 vendored or use Lambda layers; here we use a pure
    # standard-library trick: connect via Aurora Data API. Postgres on RDS
    # also supports Data API since 2024. The lab uses BackendQuery via the
    # rds-data client.
    import socket, struct
    # SIMPLIFIED: read the bronze rows via psycopg2 if available; else
    # fall back to invoking a connectivity probe.
    try:
        import psycopg2
        conn = psycopg2.connect(
            host=secret["host"], port=secret["port"], dbname=secret["database"],
            user=secret["username"], password=secret["password"],
        )
        cur = conn.cursor()
        cur.execute(query)
        cols = [d[0] for d in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchall()]
        cur.close()
        conn.close()
    except ImportError:
        return {{"name": name, "rows": 0, "warning": "psycopg2 not in Lambda runtime; deploy with layer"}}

    wm_key = f"framework/watermarks/{{name}}.json"
    try:
        last_event_id = json.loads(S3.get_object(Bucket=BUCKET, Key=wm_key)["Body"].read())["last_event_id"]
    except S3.exceptions.NoSuchKey:
        last_event_id = "0"
    new_rows = [r for r in rows if r["event_id"] > last_event_id]
    if new_rows:
        values = ",".join(
            f"({{_q(r['event_id'])}}, {{_q(r.get('source_system', name))}}, {{int(r.get('amount', 0))}}, {{_q(r.get('raw_path', ''))}}, "
            f"'{{time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}}')"
            for r in new_rows
        )
        _run_athena(f"INSERT INTO {{DB}}.{{TABLE}} (event_id, source_system, amount, raw_path, ingest_ts) VALUES {{values}}")
    if new_rows:
        max_id = max(r["event_id"] for r in new_rows)
        S3.put_object(Bucket=BUCKET, Key=wm_key, Body=json.dumps({{"last_event_id": max_id}}).encode("utf-8"))
    return {{"name": name, "rows": len(new_rows)}}
'''

DISPATCH_CODE = f'''
import json
import boto3
import yaml

S3 = boto3.client("s3")
LAMBDA = boto3.client("lambda")
BUCKET = "{BUCKET_NAME}"
FN_BY_TYPE = {{
    "sftp": "{SFTP_FN}",
    "rest": "{REST_FN}",
    "rds": "{RDS_FN}",
}}


def handler(event, context):
    config = yaml.safe_load(S3.get_object(Bucket=BUCKET, Key="framework/config.yaml")["Body"].read())
    results = []
    for source in config["sources"]:
        fn = FN_BY_TYPE[source["type"]]
        resp = LAMBDA.invoke(FunctionName=fn, Payload=json.dumps(source).encode())
        results.append(json.loads(resp["Payload"].read()))
    return {{"sources_processed": len(results), "results": results}}
'''


def _zip(code):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("index.py", code)
    buf.seek(0)
    return buf.read()


# YOUR CODE: deploy each of the four Lambdas (SFTP_FN, REST_FN, RDS_FN, DISPATCH_FN)
# using lambda_client.create_function with role=framework_role_arn, runtime="python3.13".
# Add a layer for pyyaml + psycopg2 if not in the runtime; for simplicity,
# add an inline pip-pack via the deployment package OR use the Lambda
# Klayer "psycopg2-py313" public layer arn. For the lab we accept the
# RDS Lambda may fail with "no psycopg2" and document it; the SFTP and
# REST source classes do not need extra deps.

# YOUR CODE: invoke DISPATCH_FN once and wait. Print the response.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: bronze_events has rows from all three source_system values.

def validator_3(session):
    try:
        qid = athena.start_query_execution(
            QueryString=f"SELECT source_system, count(*) c FROM {DATABASE_NAME}.{BRONZE_TABLE} GROUP BY source_system",
            WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME},
        )["QueryExecutionId"]
        deadline = time.time() + 60
        while time.time() < deadline:
            ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
            if ex["Status"]["State"] == "SUCCEEDED":
                break
            if ex["Status"]["State"] in ("FAILED", "CANCELLED"):
                return CheckpointResult("error", error_reason="Athena query failed")
            time.sleep(2)
        rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1:]
        seen = {r["Data"][0]["VarCharValue"]: int(r["Data"][1]["VarCharValue"]) for r in rows}
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Validator query failed: {exc}")
    required = ["team-a-sftp", "team-b-rest", "team-c-rds"]
    missing = [s for s in required if s not in seen]
    if missing:
        return CheckpointResult("fail", error_reason=f"Missing source_system rows for: {missing}")
    return CheckpointResult("pass")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

Each source class is a separate Lambda. The dispatcher reads YAML, maps `type` to a function name, and invokes.

</details>

<details><summary>Hint 2 (direction)</summary>

Deploy four Lambdas: SFTP_FN, REST_FN, RDS_FN, DISPATCH_FN. The RDS source class needs psycopg2 in the runtime; you can ship it via a Lambda layer or accept that the RDS source returns zero rows (the lab tolerates this if you used the layer approach). The simplest path: zip a psycopg2-binary wheel into your deployment package.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
for name, src in [(SFTP_FN, SFTP_CODE), (REST_FN, REST_CODE), (RDS_FN, RDS_CODE), (DISPATCH_FN, DISPATCH_CODE)]:
    try:
        lambda_client.create_function(
            FunctionName=name, Runtime="python3.13", Role=framework_role_arn,
            Handler="index.handler", Code={"ZipFile": _zip(src)},
            Timeout=180, MemorySize=512, Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
            Layers=["arn:aws:lambda:us-east-1:336392948345:layer:AWSSDKPandas-Python313:1"],
        )
    except lambda_client.exceptions.ResourceConflictException:
        pass

resp = lambda_client.invoke(FunctionName=DISPATCH_FN, Payload=b"{}")
print(json.loads(resp["Payload"].read()))
```

For psycopg2 in Python 3.13, use a public layer (search "psycopg2-py313 layer" on Lambda layers; AWS doesn't ship one natively).

</details>


**Wallet check.** Four Lambda deploys: free. Dispatcher run plus three source-class invocations: free. Iceberg writes from the bronze inserts: ~$0.10. Session total so far: ~10 cents (or ~10 cents + RDS hourly).

## Task 4: Idempotency. Re-run with no upstream changes; delta is zero.

Re-trigger the dispatcher. Each source class reads its watermark from `s3://<bucket>/framework/watermarks/<name>.json`, fetches only new data, and writes nothing if there is no new data.


In [ ]:
# NBVAL_SKIP
# Task 4: idempotency.

def count_bronze():
    qid = athena.start_query_execution(
        QueryString=f"SELECT count(*) FROM {DATABASE_NAME}.{BRONZE_TABLE}",
        WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME},
    )["QueryExecutionId"]
    deadline = time.time() + 60
    while time.time() < deadline:
        ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        if ex["Status"]["State"] == "SUCCEEDED":
            return int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
        if ex["Status"]["State"] in ("FAILED", "CANCELLED"):
            raise RuntimeError("Athena query failed")
        time.sleep(2)
    raise TimeoutError()


# YOUR CODE: capture pre count, re-invoke dispatcher, capture post count,
# store post - pre in IDEMPOTENCY_DELTA.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: IDEMPOTENCY_DELTA == 0.

def validator_4(session):
    delta = globals().get("IDEMPOTENCY_DELTA")
    if delta is None:
        return CheckpointResult("fail", error_reason="IDEMPOTENCY_DELTA not set.")
    if delta != 0:
        return CheckpointResult("fail", error_reason=f"IDEMPOTENCY_DELTA is {delta}; expected 0. Re-running with no new upstream data must produce zero new bronze rows.")
    return CheckpointResult("pass")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

Each source class persists its own watermark. The dispatcher should not re-read anything that is already in the watermark.

</details>

<details><summary>Hint 2 (direction)</summary>

Capture `count_bronze()` before re-invoking dispatch. Re-invoke. Capture `count_bronze()` after. Delta should be zero.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
pre = count_bronze()
lambda_client.invoke(FunctionName=DISPATCH_FN, Payload=b"{}")
time.sleep(30)
post = count_bronze()
IDEMPOTENCY_DELTA = post - pre
print(f"pre={pre}, post={post}, delta={IDEMPOTENCY_DELTA}")
```

</details>


**Wallet check.** Re-running dispatcher: free at lab volume. Athena count queries: pennies. Session total: ~10 cents.

## Task 5: Add a fourth source via YAML edit only

The team-d-sftp-2 seed has been waiting in `s3://<bucket>/sftp-drop-2/team-d/`. Append it to the YAML; re-trigger dispatch; confirm bronze grows. No Lambda redeploy.

The validator confirms the bronze row count grew AFTER you added the fourth source. It does NOT validate that you did not redeploy a Lambda; if you cheated by redeploying, you proved to yourself the framework is not yet a framework.


In [ ]:
# NBVAL_SKIP
# Task 5: append the fourth source.

# YOUR CODE: read current YAML, append a 4th source entry with
#   name=team-d-sftp-2, type=sftp, config={"path": "sftp-drop-2/"}
# upload the new YAML, re-trigger dispatch, capture FOURTH_SOURCE_DELTA.


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: bronze grew after the YAML-only edit.

def validator_5(session):
    delta = globals().get("FOURTH_SOURCE_DELTA")
    if delta is None:
        return CheckpointResult("fail", error_reason="FOURTH_SOURCE_DELTA not set.")
    if delta <= 0:
        return CheckpointResult("fail", error_reason=f"FOURTH_SOURCE_DELTA is {delta}; expected positive. Did the fourth source actually run?")
    # Also validate the YAML has 4 sources now.
    try:
        body = s3.get_object(Bucket=BUCKET_NAME, Key="framework/config.yaml")["Body"].read().decode("utf-8")
        cfg = yaml.safe_load(body)
        if len(cfg["sources"]) != 4:
            return CheckpointResult("fail", error_reason=f"YAML has {len(cfg['sources'])} sources; expected 4.")
    except Exception as exc:
        return CheckpointResult("fail", error_reason=f"YAML read failed: {exc}")
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

Editing the YAML adds the source. The dispatcher reads YAML at every invocation, so the new source flows in without code changes.

</details>

<details><summary>Hint 2 (direction)</summary>

Same shape as before. New entry has `name=team-d-sftp-2`, `type=sftp`, `config.path=sftp-drop-2/`. Watermark for the new name starts empty.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
body = s3.get_object(Bucket=BUCKET_NAME, Key="framework/config.yaml")["Body"].read().decode("utf-8")
cfg = yaml.safe_load(body)
cfg["sources"].append({"name": "team-d-sftp-2", "type": "sftp", "config": {"path": "sftp-drop-2/"}})
s3.put_object(Bucket=BUCKET_NAME, Key="framework/config.yaml", Body=yaml.safe_dump(cfg).encode("utf-8"))

pre = count_bronze()
lambda_client.invoke(FunctionName=DISPATCH_FN, Payload=b"{}")
time.sleep(30)
post = count_bronze()
FOURTH_SOURCE_DELTA = post - pre
```

</details>


**Wallet check.** YAML edit + dispatch re-run: free. The new SFTP read inserted ~20 rows: pennies. Session total so far: ~10 cents.

In [ ]:
# NBVAL_SKIP
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "athena_workgroup":
            client = boto3.client("athena", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except Exception:
                pass
            return
        if resource.type == "secret":
            client = boto3.client("secretsmanager", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_secret(SecretId=resource.id, ForceDeleteWithoutRecovery=True)
            except Exception:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("iceberg_table", "athena_workgroup", "secret"):
            return False
        return super().describe_resource(credentials, resource)


result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $1.50 to $3.00.** RDS db.t4g.micro is the boss line item; free if you are inside the 12-month window, otherwise ~$0.025 across the ~1.5-hour session. Iceberg writes from the dispatcher's multiple runs: ~$0.40. Lambda: free at lab volume.

## These are not graded. They are for you.

**1. Abstraction depth.** You built three source classes that share an ingest contract. If you had to onboard a fourth source type (GraphQL), which abstraction would you collapse first to minimize the new code? Which would you defend?

**2. Watermark storage.** SFTP uses a seen-files list, REST uses a cursor, RDS uses a max-id. Walk through what changes if a source publishes events out of order. Which of the three watermark schemes survives out-of-order writes and which break?

## Exam-Style Practice

**Question 1.** Your framework needs to onboard a fourth source type (Kafka). Which change set is the smallest?

A) Add a new source class Lambda + register `"kafka"` in the dispatcher's type-to-function map.

B) Refactor the dispatcher into a plugin loader that auto-discovers source classes by S3 path convention.

C) Rewrite the YAML schema to be Kafka-native.

D) Move the dispatcher to Step Functions.

<details><summary>Show answer</summary>

**A**.

- A) Right. The framework's design supports new source types via two changes: a new Lambda implementing the ingest contract + one line in the dispatcher's map. This is the minimum.
- B) Wrong. A plugin loader sounds elegant but adds complexity (auto-discovery rules, S3 listing on every dispatch, error modes when discovery fails) for a problem that does not yet exist. Premature.
- C) Wrong. The YAML schema is type-agnostic except for the type-specific config block. A Kafka source class adds its own config keys; the schema does not need rewriting.
- D) Wrong. Step Functions adds orchestration capabilities (retries, branching, durable state) but does not solve the onboarding problem. The dispatcher Lambda is already correct.

</details>

**Question 2.** Your RDS source class persists a watermark of "max event_id seen so far". A new requirement says the source must handle out-of-order INSERTs (event_id 100 inserted before event_id 50 reaches the read replica). Which is the lightest change?

A) Switch the watermark from "max event_id" to "max ingest_timestamp at source".

B) Use a tumbling window: re-scan the last 10 minutes on every run; rely on a downstream MERGE to dedupe.

C) Switch the source to a CDC-based read with full LSN tracking.

D) Add a delay-window between source replication and watermark advancement.

<details><summary>Show answer</summary>

**D**.

- A) Wrong. Source timestamps can be wrong (clock skew, transaction commit time vs row insert time). Substituting one ordering-fragile field for another is not a fix.
- B) Wrong. Re-scanning a window introduces duplicate work + downstream pressure on dedup. It hides the problem rather than solving it.
- C) Wrong. CDC with LSN tracking solves out-of-order rigorously but requires DMS or a CDC connector; this is a much larger change than the question implies.
- D) Right. Delay-window watermarking is the canonical fix: advance the watermark to "max event_id at T - delay" rather than "max event_id at T". A delay of 5-10 minutes catches typical late arrivals without re-scanning. Simple change.

</details>
